# 📘 Lesson 04 — SHAP Values (Explicando previsões individuais)
- Explainability Machine Learning — Study Companion (Moacir)
- Hybrid Learning Notebook — Study + Reference + Hands‑on

Nesta lesson, seguimos a lesson **“SHAP Values”** do curso Kaggle
Machine Learning Explainability, mas adaptando o código para que
você **não precise rodar o SHAP 0.51** no seu ambiente.

O foco aqui é:
- entender o conceito
- ver o fluxo de uso
- conectar com o cenário de readmissão hospitalar
- manter o código como referência, mesmo que não seja executado.

# 1) Objetivos da Lesson

- Entender o que são **SHAP Values** (SHapley Additive exPlanations).
- Ver como eles decompõem uma previsão em contribuições por feature.
- Relacionar SHAP com o cenário de readmissão hospitalar.
- Conectar SHAP com Permutation Importance e PDPs.
- Ter um **fluxo de código de referência**, mesmo sem rodar o SHAP.

# 2) Glossário Técnico

- **SHAP Values:** valores que medem quanto cada feature empurra a previsão
  para cima ou para baixo em relação a um baseline.
- **Baseline / Expected Value:** previsão média do modelo (sem informação
  específica de um indivíduo).
- **Contribuição da Feature:** impacto de um valor específico de uma feature
  na previsão final.
- **Explicação Local:** explicação de uma previsão específica (um paciente).
- **Explicação Global:** visão agregada das contribuições em muitos exemplos.

# 3) Mini‑Referência (API Style)

Conceitualmente, a lesson do Kaggle usa:

- `import shap`
- `explainer = shap.TreeExplainer(model)` *(API legacy)*
- `shap_values = explainer.shap_values(X_row)`
- `shap.force_plot(...)`

Como o `TreeExplainer` e o backend legacy do SHAP 0.51 estão
incompatíveis com ambientes modernos, aqui tratamos esse código
como **referência**, não como algo obrigatório de rodar.

# 4) Introdução Conceitual (Book‑Style)

Nas lessons anteriores, você viu:

- **Permutation Importance** → “Quais features importam para o modelo?”
- **PDPs** → “Como cada feature afeta a previsão média?”

Agora, a pergunta muda para:

> **“Por que o modelo fez esta previsão específica para este paciente?”**

Os **SHAP Values** respondem isso decompondo a previsão em:

- um **valor base** (previsão média do modelo);
- contribuições positivas (features que aumentam o risco);
- contribuições negativas (features que reduzem o risco).

A ideia central é:



\[
\text{previsão do paciente} \approx \text{baseline} + \sum \text{SHAP values das features}
\]



Ou seja, a soma das contribuições explica por que a previsão daquele
paciente é diferente da previsão média.

# 5) Implementação passo a passo (sem SHAP obrigatório)

## 5.1. Configuração de caminhos + importação de bibliotecas

Aqui seguimos o mesmo padrão da Lesson 02 (Permutation Importance).

In [4]:
import sys
from pathlib import Path

# Caminho absoluto do notebook
notebook_path = Path().resolve()

# Sobe diretórios até encontrar config.py
for parent in notebook_path.parents:
    if (parent / "config.py").exists():
        sys.path.append(str(parent))
        break

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from config import get_data_path, get_output_path

DATA_PATH = get_data_path()
OUTPUT_PATH = get_output_path()

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)

DATA_PATH: ../data/raw/
OUTPUT_PATH: ../outputs/


## 5.2. Carregar o dataset sintético

Usamos o mesmo dataset das lessons anteriores:
- `medical_readmissions.csv`

In [6]:
df = pd.read_csv(DATA_PATH + "medical_readmissions.csv")
df.head()

,age,gender,race,admission_type,time_in_hospital,time_in_hospital_scaled,num_procedures,num_medications,lab_tests,bmi_estimate,socks_owned,favorite_color,diabetes,hypertension,readmitted
0,62.450712,Female,Hispanic,Emergency,3,3.0,0,3,45.280360,21.610513,18,Blue,No,Yes,0
1,52.926035,Female,White,Emergency,4,4.0,5,3,32.315645,16.338189,6,Red,Yes,No,0
2,64.715328,Female,Other,Emergency,4,4.0,1,4,40.490947,23.851009,5,Blue,Yes,No,0
3,77.845448,Male,Asian,Urgent,6,6.0,3,4,26.482925,28.906129,4,Green,No,No,0
4,51.487699,Male,Black,Emergency,12,12.0,2,9,46.858905,11.503526,3,Green,Yes,No,0


## 5.3. Separar features e target

In [7]:
y = df["readmitted"]
X = df.drop(columns=["readmitted"])

X.head()

,age,gender,race,admission_type,time_in_hospital,time_in_hospital_scaled,num_procedures,num_medications,lab_tests,bmi_estimate,socks_owned,favorite_color,diabetes,hypertension
0,62.450712,Female,Hispanic,Emergency,3,3.0,0,3,45.280360,21.610513,18,Blue,No,Yes
1,52.926035,Female,White,Emergency,4,4.0,5,3,32.315645,16.338189,6,Red,Yes,No
2,64.715328,Female,Other,Emergency,4,4.0,1,4,40.490947,23.851009,5,Blue,Yes,No
3,77.845448,Male,Asian,Urgent,6,6.0,3,4,26.482925,28.906129,4,Green,No,No
4,51.487699,Male,Black,Emergency,12,12.0,2,9,46.858905,11.503526,3,Green,Yes,No


## 5.4. One‑Hot Encoding (mesmo padrão da Lesson 02)

In [ ]:
X = pd.get_dummies(X, drop_first=True)
X.head(), X.shape

## 5.5. Dividir em treino e validação

In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

X_train.shape, X_valid.shape

((960, 19), (240, 19))

## 5.6. Treinar o modelo base

Usamos novamente um **RandomForestClassifier** simples.

In [10]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=0
)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

# 6) SHAP Values — fluxo conceitual (sem depender do SHAP 0.51)

A lesson original do Kaggle usa algo assim:

```python
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_row)

shap.force_plot(explainer.expected_value[1],
                shap_values[1],
                X_row)
```

No seu ambiente, o backend legacy (`TreeExplainer` + `shap_values`)
está incompatível com as versões atuais de NumPy / scikit‑learn.

Então, em vez de forçar a execução, vamos:

- manter o **fluxo de código como referência**;
- focar na **interpretação conceitual**;
- deixar claro como você faria isso em um ambiente compatível.

## 6.1. Exemplo conceitual: explicando um paciente

Vamos escolher uma linha de validação (um paciente) e ver:
- a previsão do modelo;
- o que seria o “baseline”;
- como os SHAP Values entrariam na história.

In [11]:
# Escolher um paciente da validação
row_idx = 0
x_row = X_valid.iloc[row_idx:row_idx+1]
x_row

,age,time_in_hospital,time_in_hospital_scaled,num_procedures,num_medications,lab_tests,bmi_estimate,socks_owned,gender_Male,race_Black,race_Hispanic,race_Other,race_White,admission_type_Emergency,admission_type_Urgent,favorite_color_Green,favorite_color_Red,diabetes_Yes,hypertension_Yes
181,42.142637,8,8.0,0,8,NaN,16.093382,9,False,False,False,True,False,False,True,True,False,False,False


In [12]:
# Probabilidade prevista de readmissão para esse paciente
proba = model.predict_proba(x_row)[0, 1]
print(f"Probabilidade prevista de readmissão (paciente {row_idx}): {proba:.3f}")

Probabilidade prevista de readmissão (paciente 0): 0.025


In [13]:
# Probabilidade média (baseline) no conjunto de validação
proba_mean = model.predict_proba(X_valid)[:, 1].mean()
print(f"Probabilidade média (baseline) no conjunto de validação: {proba_mean:.3f}")

Probabilidade média (baseline) no conjunto de validação: 0.101


Conceitualmente, os SHAP Values fariam:

- pegar esse **baseline** (probabilidade média);
- somar contribuições positivas/negativas de cada feature do paciente;
- chegar na probabilidade final prevista para ele.

Em um ambiente compatível com SHAP 0.51, o código seria algo como:

```python
import shap

# Criar o explainer (versão legacy, como no Kaggle)
explainer = shap.TreeExplainer(model)

# Calcular SHAP values para uma única linha
shap_values = explainer.shap_values(x_row)

# expected_value[1] = baseline para a classe "1" (readmitido)
# shap_values[1] = contribuições das features para a classe "1"
shap.initjs()
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1],
    x_row
)
```

> **Importante:**  
> No seu ambiente atual, esse código pode falhar por incompatibilidade
> de versões. Aqui ele aparece como **referência**, não como algo
> obrigatório de rodar.

## 6.2. Interpretação visual (sem rodar o SHAP)

A visualização típica (force plot / waterfall) mostra:

- um ponto de partida: **baseline** (previsão média);
- barras vermelhas: features que **aumentam** o risco de readmissão;
- barras azuis: features que **diminuem** o risco;
- o valor final: previsão do modelo para aquele paciente.

Em termos de narrativa para o médico:

- “Este paciente tem risco de readmissão de 0.73.”
- “Isso é maior que a média de 0.40.”
- “Os principais fatores que aumentam o risco são: tempo de internação alto,
  muitos medicamentos, presença de diabetes.”
- “Os fatores que reduzem o risco são: idade mais baixa, poucos procedimentos, etc.”

## 6.3. Função `patient_risk_factors` (versão conceitual)

No Kaggle, a lesson termina com uma função que:

- recebe os dados de um paciente;
- calcula SHAP Values;
- gera um gráfico explicando o risco.

Aqui, vamos montar a **assinatura** e a **lógica narrativa**, sem
depender da execução do SHAP.

In [14]:
def patient_risk_factors_conceitual(row_idx, X_valid, model):
    """
    Versão conceitual da função patient_risk_factors.

    Em um ambiente com SHAP funcionando, aqui entraríamos com:
    - cálculo de SHAP Values
    - gráfico waterfall/force plot

    Aqui, vamos:
    - mostrar a probabilidade prevista
    - comparar com a média
    - listar algumas features do paciente
    """
    x_row = X_valid.iloc[row_idx:row_idx+1]
    proba = model.predict_proba(x_row)[0, 1]
    proba_mean = model.predict_proba(X_valid)[:, 1].mean()

    print(f"Paciente {row_idx}")
    print(f"Probabilidade prevista de readmissão: {proba:.3f}")
    print(f"Probabilidade média (baseline): {proba_mean:.3f}")
    print("\nAlgumas features do paciente:")
    display(x_row.iloc[0].sort_values(ascending=False).head(10))

# Exemplo de uso
patient_risk_factors_conceitual(0, X_valid, model)

Paciente 0
Probabilidade prevista de readmissão: 0.025
Probabilidade média (baseline): 0.101

Algumas features do paciente:


age                         42.142637
bmi_estimate                16.093382
socks_owned                         9
time_in_hospital_scaled           8.0
num_medications                     8
time_in_hospital                    8
favorite_color_Green             True
race_Other                       True
admission_type_Urgent            True
admission_type_Emergency        False
Name: 181, dtype: object

> Em um ambiente com SHAP estável, você substituiria o corpo dessa função
> por chamadas a `shap.Explainer` ou `shap.TreeExplainer` + gráficos
> `shap.plots.waterfall` ou `shap.force_plot`.

# 7) Observações pedagógicas do Copilot

- SHAP Values são uma ferramenta **local**: explicam uma previsão específica.
- Eles complementam:
  - **Permutation Importance** → “o que importa em média”
  - **PDPs** → “como a previsão média muda com a feature”
- No cenário de readmissão hospitalar, SHAP é ideal para:
  - explicar por que um paciente foi classificado como alto risco;
  - apoiar decisões médicas com transparência;
  - comunicar fatores de risco de forma clara.
- Por limitações de compatibilidade (SHAP 0.51 + NumPy + sklearn),
  tratamos o código de SHAP aqui como **referência conceitual**,
  não como algo obrigatório de rodar.
- Em um ambiente controlado (como o Kaggle original), o fluxo de código
  funciona exatamente como mostrado na lesson oficial.

# 8) Conclusão

Nesta Lesson 04, você:

- entendeu o conceito de **SHAP Values**;
- viu como eles decompõem uma previsão em contribuições por feature;
- conectou SHAP com o cenário de readmissão hospitalar;
- viu o fluxo de código da lesson do Kaggle, mesmo sem rodar o SHAP 0.51;
- reforçou a diferença entre:
  - **insights globais** (Permutation Importance, PDPs)
  - **insights locais** (SHAP).

Mensagem principal:

> SHAP Values respondem:  
> **“Por que o modelo fez esta previsão específica para este paciente?”**

No fluxo do curso de Explainability, o próximo passo é usar SHAP
em conjunto com outras técnicas para construir análises completas
em cenários reais.